<a href="https://colab.research.google.com/github/KoyaChanS/DPRK_ITW_financial_behaviour_analysis-/blob/DPRK_ITW_multi_hop_analysis/DPRK_ITW_multi_hop_for_git.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Data Acquisition and Preprocessing:
This section focuses on acquiring the raw data and preparing it for analysis, including timestamp conversions and ETH to USD price enrichment.

In [ ]:
import duckdb as db


## Miscellaneous functions
def display_table_title(titleText:str) -> str:
  TITLE_SEPARATOR = "="*50
  return f"{TITLE_SEPARATOR} {titleText} {TITLE_SEPARATOR}"

#--------------------READ THE CSV FILE --------------------

hops_df = db.sql('''
SELECT *
FROM read_csv('/content/sample_data/hops_labelled.csv');''').df()

#--------------------CONVERTING UNIX TIMESTAMP TO HUMAN READABLE TIMESTAMP--------------------

hops_raw_df = db.sql('''
SELECT *
EXCLUDE(first_seen, last_seen),
strftime(to_timestamp(first_seen),'%Y-%m-%d %H:%M:%S') AS first_seen,
strftime(to_timestamp(last_seen), '%Y-%m-%d %H:%M:%S') AS last_seen
FROM hops_df
;''').df()
display(hops_raw_df)

print(display_table_title("hops_raw_df"))

max_timestamp = db.sql('''
SELECT MAX(first_seen) AS max_first_seen,
FROM hops_raw_df;''').df()['max_first_seen'].iloc[0]
display(max_timestamp)

min_timestamp = db.sql('''
SELECT MIN(first_seen) AS min_first_seen,
FROM hops_raw_df;''').df()['min_first_seen'].iloc[0]
display(min_timestamp)

#--------------------ENRICHING DATA WITH YFINANCE - CONVERTING ETH VALUES TO USD--------------------

import yfinance as yf
import pandas as pd

eth_prices_df = yf.Ticker("ETH-USD")
eth_prices_df = eth_prices_df.history(start="2020-08-17", end="2026-03-09", interval="1d")
eth_prices_df.reset_index(inplace=True)
display(eth_prices_df)

print(display_table_title("eth_prices_df"))

#--------------------CHOOSING THE 'CLOSE' VALUE FOR THE DAY--------------------

eth_usd_price_df = db.sql('''
SELECT Date,
Close AS yfinance_price
FROM eth_prices_df;''').df()
display(eth_usd_price_df)

print(display_table_title("eth_usd_price_df"))


#--------------------MERGING THE USD INFO TO THE hops_raw_df WITH first_seen USING LEFT JOIN--------------------

hops_raw_price_df = db.sql('''
SELECT *
FROM hops_raw_df t1
INNER JOIN eth_usd_price_df t2
ON CAST(t1.first_seen AS DATE) = CAST(t2.Date AS DATE);''').df()
display(hops_raw_price_df)
print(display_table_title("hops_raw_price_df"))

print(hops_raw_price_df.dtypes)
print()



#--------------------Preview the timestamp/datetime column specifically--------------------
print(hops_raw_price_df[["from_address", "to_address", "hop_depth", "first_seen"]].head(10))
print(hops_raw_price_df["first_seen"].head(5).tolist())

#--------------------GENERATING A FOUNDATION TABLE FOR ALL SQL QUEREIS--------------------
print(display_table_title("foundation_table_df"))

foundation_table_df = db.sql('''
SELECT *,
value_eth * yfinance_price AS USD_price_per_ETH,
FROM hops_raw_price_df;''').df()
display(foundation_table_df)


print(
    foundation_table_df
    .groupby("to_entity_type")["to_address"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"to_address": "unique_addresses"})
)
#--------------------CHECKING THE NO. OF NON-NULL ENTITIES--------------------
print(foundation_table_df["to_entity_type"].value_counts(dropna=False))




In [ ]:
foundation_table_df_1 = db.sql('''
SELECT to_address,
hop_depth,
to_entity_type
FROM foundation_table_df
WHERE to_entity_type = 'SANCTIONED'
GROUP BY to_address,hop_depth, to_entity_type;''').df()
display(foundation_table_df_1)

## 2. Hop Trail & Wallet Behavioral Analytics:
This section analyzes the flow of funds across different hop depths and examines wallet transaction behavior within the network

In [ ]:
#--------------------RQ: for every dollar that left a DPRK ITW payment wallet address, how much can we trace at every hop and--------------------
#--------------------what does the number of transactions per hop tell us?--------------------

print(display_table_title("hop_trail_summary_df"))
hop_trail_summary_df = db.sql('''
SELECT
hop_depth,
tx_count,

COUNT(*) AS count,
COUNT(from_address) AS count_from_address,
COUNT(to_address) AS count_to_address,
COUNT(DISTINCT from_address) AS unique_sender_address,
COUNT(DISTINCT to_address) AS unique_reciever_address,
SUM(value_eth) AS sum_eth,
SUM(USD_price_per_ETH) AS USD_price_per_ETH
FROM foundation_table_df
GROUP BY hop_depth, tx_count
ORDER BY hop_depth;''').df()
display(hop_trail_summary_df)

print(display_table_title("percentage_hop_trail_summary_df"))
percentage_hop_trail_summary_df = db.sql('''
SELECT
hop_depth,
tx_count,
ROUND(USD_price_per_ETH * 100 / SUM(USD_price_per_ETH) OVER (PARTITION BY hop_depth), 2) AS percentage_of_eth_usd_per_hop_depth, -- CalculatIng the percentage of USD value within its hop_depth partition
ROUND(count * 100 / SUM(count) OVER (PARTITION BY hop_depth), 2) AS percentage_of_hops_per_hop_depth -- Calculating the percentage of hops (count) within its hop_depth partition
FROM hop_trail_summary_df
ORDER BY hop_depth, tx_count;''').df()
display(percentage_hop_trail_summary_df)

print(display_table_title("tx_count_behaviour_df"))
tx_count_behaviour_df = db.sql('''
SELECT
hop_depth,
tx_count,
sum_eth,
USD_price_per_ETH,
count,
CASE
WHEN tx_count = 1 THEN 'single_use'
WHEN tx_count <= 5 THEN 'low_frequency'
WHEN tx_count <= 25 THEN 'medium_frequency'
ELSE  'high_frequency'
END AS tx_count_behaviour
FROM hop_trail_summary_df
ORDER BY hop_depth, tx_count;''').df()
display(tx_count_behaviour_df)


In [ ]:
import plotly.express as px
import plotly.graph_objects as go

#color plaette per behaviour--------------------
color_map = {
    'single_use': '#E74C3C',
    'low_frequency': '#2471A3',
    "medium_frequency": "#F1C40F",
    "high_frequency": "#2ECC71"
}

#Enforcing a logical stack order--------------------
behaviour_order = ["high_frequency", "medium_frequency", "low_frequency","single_use"]

#Build one bar trace per behaviour category--------------------

fig = go.Figure()

for behaviour in behaviour_order:
  subset = vizB_df[vizB_df['tx_count_behaviour'] == behaviour]
  fig.add_trace(go.Bar(
    name=behaviour.replace("_","").title(),
    x=subset["hop_depth"],
    y=subset["total_count"],
    marker_color=color_map[behaviour],
    hovertemplate=(
        "<b>%{fullData.name}</b><br>"
        "Hop Depth: %{x}<br>"
        "Edge Count: %{y:,}<br>"
        "<extra></extra>"
      )
  ))

fig.update_layout(
    barmode='stack',
    title=dict(
        text="Transaction Behaviour by Hop Depth",
        subtitle=dict(
            text="Edge count broken down by wallet reuse frequency - DPRK ITW network"
        ),
        font=dict(size=16,color="#2C3E50",family="Arial"),
        x=0.5,
        xanchor="center"
    ),
    xaxis=dict(
        title = "Hop Depth",
        tickmode= "linear",
        tick0=1,
        dtick=0,
        showgrid=False,
        linecolor="#BDC3C7",
        tickfont=dict(color="#2C3E50")
    ),
    yaxis=dict(
        title = "Number of Edges",
        showgrid=True,
        gridcolor="#ECF0F1",
        linecolor="#BDC3C7",
        tickfont=dict(color="#2C3E50")
    ),
    legend = dict(
        title=dict(text="Wallet Behaviour"),
        orientation= "v",
        x          = 1.02,
        y          = 1,
        bgcolor    = "#FDFEFE",
        bordercolor= "#BDC3C7",
        borderwidth= 1
    ),
    paper_bgcolor = "#FDFEFE",
    plot_bgcolor  = "#FDFEFE",
    height        = 500,
    width         = 800,
    margin        = dict(t=80, b=60, l=60, r=150),

    annotations = [dict(
        text      = "Single-use wallets dominate at every hop depth,consistent with automated throwaway wallet architecture",
        xref      = "paper", yref="paper",
        x=0.5, y=-0.17,
        showarrow = False,
        font      = dict(size=11, color="#7F8C8D", family="Arial"),
        align     = "center"
    )]
)

fig.show()
fig.write_html('/content/sample_data/html/figure_1.html')


## 3. Inter-Hop Timing Analysis:
This section analyzes the time taken for funds to move between consecutive hop depths identifying patterns in transaction speed.

In [ ]:
#--------------------RQ: what is the time between each hop depth--------------------

#---------------------STEP  1: CORE AGGREGATION---------------------

print(display_table_title("hop_intervals_df"))

hop_intervals_df = db.sql('''
SELECT
  root_id,
  hop_depth,
  from_address,
  to_address,
  MIN(CAST(first_seen AS TIMESTAMP)) AS earliest_seen,
  MAX(CAST(last_seen AS TIMESTAMP)) AS lastest_seen,
  SUM(value_eth) AS sum_eth,
  SUM(USD_price_per_ETH) AS USD_price_per_ETH
FROM foundation_table_df
GROUP BY root_id,hop_depth, from_address, to_address
QUALIFY ROW_NUMBER() OVER (
  PARTITION BY hop_depth, from_address, to_address
  ORDER BY MIN(CAST(first_seen AS TIMESTAMP)) ASC) = 1
;''').df()
display(hop_intervals_df)


#---------------------STEP 2: SELF JOIN TO GET CONSEQUTIVE HOP PAIRS---------------------

print(display_table_title("hop_intervals_df_1"))
hop_intervals_df_1 = db.sql('''
SELECT
  t1.root_id,
  t1.from_address,
  t1.to_address AS relay_address,
  t2.to_address AS final_address,
  t1.hop_depth AS hop_from,
  t2.hop_depth AS hop_to,
  t1.earliest_seen AS hop_from_time,
  t2.earliest_seen AS hop_to_time,
  date_diff('hour', t1.earliest_seen, t2.earliest_seen) AS hours_between_hops,
  date_diff('day', t1.earliest_seen, t2.earliest_seen) AS days_between_hops,
  ROUND(t1.sum_eth, 4) AS value_eth,
  ROUND(t1.USD_price_per_ETH, 2) AS total_usd
FROM hop_intervals_df t1
JOIN hop_intervals_df t2
  ON t1.root_id = t2.root_id
  AND t2.hop_depth = t1.hop_depth + 1
  AND t1.to_address = t2.from_address
WHERE date_diff('second', t1.earliest_seen, t2.earliest_seen) >=0

ORDER BY  t1.hop_depth, t1.root_id;''').df()
display(hop_intervals_df_1)

sanity_check_df = db.sql('''
SELECT hop_depth, COUNT(*) AS rows, COUNT(DISTINCT from_address || to_address) AS unique_pairs
FROM hop_intervals_df
GROUP BY hop_depth
ORDER BY hop_depth;''').df()
display(sanity_check_df)


#---------------------STEP 3: CLASSIFY SPEED + GROUP IN ONE QUERY---------------------

print(display_table_title("grouped_figure_df"))
grouped_figure_df = db.sql('''
WITH classification AS (
SELECT *,
CASE
WHEN date_diff('second', hop_from_time, hop_to_time) < 60 THEN 'instantaneous'
WHEN date_diff('hour', hop_from_time, hop_to_time) < 1 THEN 'under_an_hour'
WHEN date_diff('hour', hop_from_time, hop_to_time) < 24  THEN 'under_24_hours'
WHEN date_diff('day', hop_from_time, hop_to_time) < 7 THEN 'under_a_week'
ELSE 'over_a_week'
END AS time_between_hops
FROM hop_intervals_df_1
)
SELECT hop_from,	hop_to, time_between_hops,
COUNT(*) AS COUNT
FROM classification
GROUP BY hop_from,	hop_to, time_between_hops
ORDER BY hop_from, count asc;''').df()
display(grouped_figure_df)


print(f"Total hop pairs analysed: {len(grouped_figure_df)}")
print(f"\nSpeed distribution:")


print(display_table_title("ratio_df"))
ratio_df = db.sql('''
SELECT
concat("hop_from",'-',"hop_to") AS group_hop,
time_between_hops,
COUNT
FROM grouped_figure_df
;''').df()
display(ratio_df)


print(display_table_title("ratio_df_1"))
ratio_df_1 = db.sql('''
WITH r1 AS(
SELECT
SUM(COUNT) AS total_pairs,
time_between_hops,
FROM ratio_df
GROUP BY time_between_hops
)
SELECT
time_between_hops,
SUM(total_pairs) OVER () AS sum_total_pairs,
ROUND(total_pairs*100/SUM(total_pairs) OVER(), 4) AS ratio
FROM r1;''').df()
display(ratio_df_1)



In [ ]:
#---------------------STEP 4: VISUALISATION---------------------

import plotly.graph_objects as go

#FACETED PIE CHARTS---------------------
from plotly.subplots import make_subplots
import plotly.graph_objects as go

#Colour map consistent across all pies---------------------
colour_map = {
    "instantaneous":  "#2ECC71",
    "under_an_hour":  "#F1C40F",
    "under_24_hours": "#E67E22",
    "under_a_week":   "#E74C3C",
    "over_a_week":    "#922B21"
}

category_order = [
    "instantaneous",
    "under_an_hour",
    "under_24_hours",
    "under_a_week",
    "over_a_week"
]

transitions = (
    grouped_figure_df[["hop_from","hop_to"]]
    .drop_duplicates()
    .sort_values(by=["hop_from","hop_to"])
    .reset_index(drop=True)
)

n=len(transitions)

#Makeing Subplots---------------------
fig = make_subplots(
    rows  = 1,
    cols  = n,
    specs = [[{"type": "pie"}] * n],
    subplot_titles = [
        f"Hop {row.hop_from} → Hop {row.hop_to}"
        for row in transitions.itertuples()]
)

for i, row in transitions.iterrows():
    hop1 = row["hop_from"]
    hop2 = row["hop_to"]

    subset = (
        grouped_figure_df[
            (grouped_figure_df["hop_from"] == hop1) &
            (grouped_figure_df["hop_to"] == hop2)
        ]
        .groupby("time_between_hops")["COUNT"]
        .sum()
        .reindex(category_order, fill_value=0)
        .reset_index()
    )


    fig.add_trace(
        go.Pie(
            labels      = subset["time_between_hops"].tolist(),
            values      = subset["COUNT"].tolist(),
            hole        = 0.35,
            name        = f"Hop {hop1} → Hop {hop2}",
            marker      = dict(
                colors = [colour_map[c] for c in subset["time_between_hops"]],
                line   = dict(color="white", width=1.5)
            ),
            textinfo      = "percent",
            hovertemplate = (
                "<b>%{label}</b><br>"
                "Count: %{value}<br>"
                "Share: %{percent}<br>"
                "<extra></extra>"
            ),
            showlegend = True if i == 0 else False
        ),
        row=1, col=i+1
    )

fig.update_layout(
    title = dict(
        text    = "Speed of Fund Movement by Hop Transition",
        subtitle= dict(
            text = "Distribution of time between consecutive hops — DPRK ITW network"
        ),
        font    = dict(size=16, color="#2C3E50", family="Arial"),
        x       = 0.5,
        xanchor = "center"
    ),
    legend = dict(
        orientation = "h",
        x           = 0.5,
        xanchor     = "center",
        y           = -0.15,
        bgcolor      = "#FDFEFE",
        bordercolor = "#BDC3C7",
        borderwidth = 1
    ),
    paper_bgcolor = "#FDFEFE",
    height        = 420,
    width         = 300 * n,
    font          = dict(size=11, color="#2C3E50", family="Arial"),
    margin        = dict(t=100, b=80, l=20, r=20),
    annotations   = [dict(
        text      = "Each chart shows the speed distribution of fund movement arriving at that hop depth",
        xref      = "paper", yref="paper",
        x=0.5, y=-0.14,
        showarrow = False,
        font      = dict(size=10, color="#7F8C8D", family="Arial"),
        align     = "center"
    )]
)

fig.show()
fig.write_html('/content/sample_data/html/figure_2.html')


## 4. Mid-month Clustering:
This section explores the temporal distribution of transactions across hop depths, focusing on daily patterns like mid-month clustering.

In [ ]:
print(display_table_title("hop_1_outbound_df"))

hop_1_outbound_df = db.sql('''
SELECT
root_id,
  DAYOFMONTH(CAST(first_seen AS DATE)) AS day_of_month,
  MONTH(CAST(first_seen AS DATE)) AS month,
  YEAR(CAST(first_seen AS DATE)) AS year,
  SUM(USD_price_per_ETH) AS usd_value_hop1,
  COUNT(*) AS tx_count,
  COUNT(DISTINCT to_address) AS unique_recipients
FROM foundation_table_df
WHERE hop_depth = 1
GROUP BY day_of_month, month, year, root_id
ORDER BY day_of_month, month, year;'''
).df()
display(hop_1_outbound_df)

print(display_table_title("hop_4_inbound_df"))

hop_4_inbound_df = db.sql('''
SELECT
root_id,
  DAYOFMONTH(CAST(first_seen AS DATE)) AS day_of_month,
  MONTH(CAST(first_seen AS DATE)) AS month,
  YEAR(CAST(first_seen AS DATE)) AS year,
  SUM(USD_price_per_ETH) AS usd_value_hop4,
  COUNT(*) AS tx_count,
  COUNT(DISTINCT from_address) AS unique_senders,
  COUNT(DISTINCT to_address) AS unique_recipients
FROM foundation_table_df
WHERE hop_depth = 4
GROUP BY day_of_month, month, year, root_id
ORDER BY day_of_month, month, year;'''
).df()
display(hop_4_inbound_df)


#avg lag(time) between hop1 and hop4 for the same root_id in the same month

print(display_table_title("lag_df"))

lag_df = db.sql('''
SELECT
  root_id,
  MONTH(CAST(first_seen AS DATE)) AS month,
  YEAR(CAST(first_seen AS DATE)) AS year,
  AVG(CASE WHEN hop_depth = 1 THEN DAYOFMONTH(CAST(first_seen AS DATE)) END) AS avg_day_hop1,
  AVG(CASE WHEN hop_depth = 4 THEN DAYOFMONTH(CAST(first_seen AS DATE)) END) AS avg_day_hop4,
  AVG(CASE WHEN hop_depth = 4 THEN (USD_price_per_ETH)END) AS avg_usd_value_hop4
FROM foundation_table_df
WHERE hop_depth IN (1,4)
GROUP BY root_id, month, year
HAVING avg_day_hop1 IS NOT NULL AND avg_day_hop4 IS NOT NULL;''').df()
display(lag_df)


#root_id + month combinations where both a hop_0 outbound and a hop_4 inbound fire within the same calendar month,
#and the hop_4 value lands in that $2K–$5K salary range

print(display_table_title("pairing_df"))

pairing_df = db.sql('''
SELECT
  t1.root_id,
  t1.root_label,
  t1.month,
  t1.year,
  t1.avg_day_hop1,
  t2.avg_day_hop4,
  ROUND(t2.avg_day_hop4 - t1.avg_day_hop1, 1) AS lag_days,
  t1.usd_value_hop1,
  t2.usd_value_hop4,
  t2.tx_count,
  t2.unique_recipients
FROM(
  SELECT
    root_id,
    root_label,
    MONTH(CAST(first_seen AS DATE)) AS month,
    YEAR(CAST(first_seen AS DATE)) AS year,
    AVG(DAYOFMONTH(CAST(first_seen AS DATE))) AS avg_day_hop1,
    SUM(USD_price_per_ETH) AS usd_value_hop1
  FROM foundation_table_df
  WHERE hop_depth = 1
  GROUP BY root_id, root_label, month, year
)t1
INNER JOIN(
  SELECT
    root_id,
    MONTH(CAST(first_seen AS DATE)) AS month,
    YEAR(CAST(first_seen AS DATE)) AS year,
    AVG(DAYOFMONTH(CAST(first_seen AS DATE))) AS avg_day_hop4,
    SUM(USD_price_per_ETH) AS usd_value_hop4,
    COUNT(*) AS tx_count,
    COUNT(DISTINCT to_address) AS unique_recipients
  FROM foundation_table_df
  WHERE hop_depth = 4
  GROUP BY root_id, month, year
)t2

ON t1.root_id = t2.root_id -----same suspicious wallet
AND t1.month = t2.month
AND t1.year = t2.year

WHERE
t2.usd_value_hop4 BETWEEN 2000 AND 5000 ----salary range filter
AND t2.avg_day_hop4 - t1.avg_day_hop1 BETWEEN 0 AND 7 ---acounting for the lag days

ORDER BY

t1.root_id, t1.year, t1.month;''').df()
display(pairing_df)

print(display_table_title("pairing_df_1"))

pairing_df_1 = db.sql('''
SELECT
  t1.root_id,
  t1.root_label,
  t1.month,
  t1.year,
  t1.day_hop1,
  t2.day_hop4,
  ROUND(t2.day_hop4 - t1.day_hop1, 1) AS lag_days,
  t1.usd_value_hop1,
  t2.usd_value_hop4,
  t2.tx_count,
  t2.unique_recipients
FROM(
  SELECT
    root_id,
    root_label,
    MONTH(CAST(first_seen AS DATE)) AS month,
    YEAR(CAST(first_seen AS DATE)) AS year,
    DAYOFMONTH(CAST(first_seen AS DATE)) AS day_hop1,
    SUM(USD_price_per_ETH) AS usd_value_hop1
  FROM foundation_table_df
  WHERE hop_depth = 1
  GROUP BY root_id, root_label, month, year, day_hop1
)t1
INNER JOIN(
  SELECT
    root_id,
    MONTH(CAST(first_seen AS DATE)) AS month,
    YEAR(CAST(first_seen AS DATE)) AS year,
    DAYOFMONTH(CAST(first_seen AS DATE)) AS day_hop4,
    SUM(USD_price_per_ETH) AS usd_value_hop4,
    COUNT(*) AS tx_count,
    COUNT(DISTINCT to_address) AS unique_recipients
  FROM foundation_table_df
  WHERE hop_depth = 4
  GROUP BY root_id, month, year, day_hop4
)t2

ON t1.root_id = t2.root_id -----same suspicious wallet
AND t1.month = t2.month
AND t1.year = t2.year

WHERE
t2.usd_value_hop4 BETWEEN 2000 AND 5000 ----salary range filter
AND t2.day_hop4 - t1.day_hop1 BETWEEN 0 AND 7 ---acounting for the lag days

ORDER BY

t1.root_id, t1.year, t1.month;''').df()
display(pairing_df_1)

pairing_df_2= db.sql('''
SELECT
  t1.root_id,
  t1.month,
  t1.year,
  t1.avg_day_hop1,
  t2.avg_day_hop4,
  (t2.avg_day_hop4 - t1.avg_day_hop1) AS lag_days,
  t2.total_settled_usd -- Focus on the final received amount
FROM (
  SELECT
    root_id, month, year,
    AVG(day_of_month) as avg_day_hop1
  FROM hop_1_outbound_df
  GROUP BY 1,2,3
) t1
INNER JOIN (
  SELECT
    root_id, month, year,
    AVG(day_of_month) as avg_day_hop4,
    SUM(usd_value_hop4) as total_settled_usd
  FROM hop_4_inbound_df
  GROUP BY 1,2,3
  HAVING SUM(usd_value_hop4) BETWEEN 2000 AND 10000 -- The key filter
) t2
ON t1.root_id = t2.root_id
AND t1.month = t2.month
AND t1.year = t2.year
WHERE (t2.avg_day_hop4 - t1.avg_day_hop1) BETWEEN 0 AND 14;''').df()
display(pairing_df_2)

chain_of_custody_df=db.sql('''
WITH hop_steps AS (
    SELECT
        root_id,
        hop_depth,
        CAST(first_seen AS DATE) as event_date,
        DAYOFMONTH(CAST(first_seen AS DATE)) as day_num,
        USD_price_per_ETH as usd_val
    FROM foundation_table_df
)
SELECT
    t1.root_id,
    t1.hop_depth AS from_hop,
    t2.hop_depth AS to_hop,
    t1.day_num AS start_day,
    t2.day_num AS end_day,
    DATEDIFF('day', t2.event_date, t1.event_date) AS step_lag,
    t2.usd_val AS transfer_val
FROM hop_steps t1
JOIN hop_steps t2
  ON t1.root_id = t2.root_id
  AND t2.hop_depth = t1.hop_depth + 1 -- This links 1 to 2, 2 to 3, etc.
WHERE
    t1.day_num BETWEEN 10 AND 15 -- Checking if the "From" side is in the cluster
    AND t2.day_num BETWEEN 10 AND 15 -- Checking if the "To" side stays in the cluster
    AND t2.usd_val BETWEEN 2000 AND 5000 -- The salary profile
    AND DATEDIFF('day', t2.event_date, t1.event_date) BETWEEN 0 AND 3;''').df() #Expecting fast movement between hops
display(chain_of_custody_df)

In [ ]:
#1. Calculate Average Day per Month for Hop 1
hop1_trend = hop_1_outbound_df.groupby(['year', 'month'])['day_of_month'].mean().reset_index()
hop1_trend['hop'] = 'Hop 1 (Dispatch)'

#2. Calculate Average Day per Month for Hop 4
hop4_trend = hop_4_inbound_df.groupby(['year', 'month'])['day_of_month'].mean().reset_index()
hop4_trend['hop'] = 'Hop 4 (Receipt)'

#3. Merge for plotting
trend_df = pd.concat([hop1_trend, hop4_trend])
trend_df['date'] = pd.to_datetime(trend_df[['year', 'month']].assign(day=1))

import plotly.express as px

fig = px.line(
    trend_df,
    x="date",
    y="day_of_month",
    color="hop",
    markers=True,
    title="Evidence of Coordinated Timing: Hop 1 vs. Hop 4",
    labels={"day_of_month": "Average Day of Month", "date": "Timeline"},
    color_discrete_map={
        "Hop 1 (Dispatch)": "#3498db",
        "Hop 4 (Receipt)": "#e67e22"
    }
)

#Add the Mid-Month Cluster Zone (Y-axis highlight)
fig.add_hrect(
    y0=10, y1=15,
    fillcolor="rgba(231, 76, 60, 0.2)",
    line_width=0,
    annotation_text="Expected Mid-Month Clustering",
    annotation_position="top left"
)

fig.update_layout(
    yaxis=dict(range=[0, 31], dtick=5),
    hovermode="x unified",
    template="plotly_white"
)

fig.show()
fig.write_html('/content/sample_data/html/figure_3.html')


In [ ]:
avg_day_hop4 = db.sql('''
SELECT root_id,
AVG(DAYOFMONTH(CAST(first_seen AS DATE))) AS avg_day_hop4
FROM foundation_table_df
WHERE hop_depth = 4
GROUP BY root_id;''').df()
display(avg_day_hop4)

## 5. Network Intersections: Shared Infrastructure:
This section identifies common wallet addresses that receive funds from multiple sources.

In [ ]:
#--------------------RQ: identifying which addresses recieve funds from the most root ids?--------------------
#the point is to identify which wallet/s end up being the shared by these root ids--------------------

print(display_table_title("shared_infrastructure_df"))

shared_infrastructure_df = db.sql('''
SELECT
to_address,
from_address,
COUNT(DISTINCT from_address) AS unique_sender_address,
COUNT(DISTINCT to_address) AS unique_recipient_address,
COUNT(DISTINCT root_id) AS unique_root_id,
COUNT(*) AS count,
SUM(value_eth) AS sum_eth,
SUM(USD_price_per_ETH) AS USD_price_per_ETH,
MIN(hop_depth) AS min_hop_depth,
MAX(hop_depth) AS max_hop_depth,
FROM foundation_table_df
GROUP BY to_address, from_address
HAVING COUNT(DISTINCT root_id) > 1
ORDER BY count DESC;''').df()
display(shared_infrastructure_df)


#Combining consolidated address--------------------

consolidated_addresses_df = db.sql('''
SELECT *
FROM read_csv('/content/sample_data/consolidatedaddresses.csv');''').df()

print(display_table_title("combined_hops_df"))
combined_hops_df = db.sql('''
SELECT t1.consolidatedaddresses,
t2.unique_root_id,
t2.unique_sender_address,
t2.count,
t2.to_address,
t2.from_address,
t2.sum_eth,
t2.USD_price_per_ETH,
t2.min_hop_depth,
t2.max_hop_depth
FROM consolidated_addresses_df t1
INNER JOIN shared_infrastructure_df t2
ON lower(t1.consolidatedaddresses) = lower(t2.to_address)
OR lower (t1.consolidatedaddresses) = lower(t2.from_address);''').df()
display(combined_hops_df)


#--------------------RQ:At what depth does the DOJ designated consolidated addressess appear--------------------
print(display_table_title("consolidated_address_depth_df"))
consolidated_address_depth_df = db.sql('''
SELECT t1.consolidatedaddresses,
t2.root_id,
t2.to_address,
t2.from_address,
t2.hop_depth,
t2.value_eth,
t2.tx_count,
t2.recipient_count,
t2.fan_out_capped,
t2.first_seen,
t2.last_seen,
t2.Date,
t2.yfinance_price,
t2.USD_price_per_ETH
FROM consolidated_addresses_df t1
INNER JOIN foundation_table_df t2
ON lower(t1.consolidatedaddresses) = lower(t2.to_address)
OR lower(t1.consolidatedaddresses) = lower(t2.from_address)
WHERE root_id IS NOT NULL;''').df()
display(consolidated_address_depth_df)

print(f"Unique consolidation addresses: {consolidated_address_depth_df['consolidatedaddresses'].nunique()}")
print(f"Unique root wallets:            {consolidated_address_depth_df['root_id'].nunique()}")
print(f"Hop depths seen:                {sorted(consolidated_address_depth_df['hop_depth'].unique().tolist())}")

print(display_table_title("unique_consolidated_df"))
unique_consolidated_df = db.sql('''
SELECT DISTINCT consolidatedaddresses
FROM consolidated_address_depth_df
ORDER BY consolidatedaddresses;
''').df()
display(unique_consolidated_df)

unique_consolidated_df_1 = db.sql('''
SELECT
    consolidatedaddresses,
    COUNT(DISTINCT root_id)          AS unique_root_ids,
    COUNT(DISTINCT hop_depth)        AS unique_hop_depths,
    MIN(hop_depth)                   AS min_hop_depth,
    MAX(hop_depth)                   AS max_hop_depth,
    SUM(value_eth)                   AS total_eth_involved,
    MIN(first_seen)                  AS first_seen,
    MAX(last_seen)                   AS last_seen
FROM consolidated_address_depth_df
GROUP BY consolidatedaddresses
ORDER BY unique_root_ids DESC;
''').df()
display(unique_consolidated_df_1)

#--------------------Path analysis ===> the behaviorual split shows which address sends value to consilidation address at what hop depth--------------------
#hop1 means the sender(root_id operators kind of knows they are sending to a consolidation address)--------------------
#other hop depths only show that there are probably different operators at different levels who know about these addresess--------------------
print(display_table_title("path_analysis_df"))
path_analysis_df = db.sql('''
SELECT
root_id,
MIN(hop_depth) AS shortest_hop_depth,
FROM consolidated_address_depth_df
GROUP BY root_id;''').df()
display(path_analysis_df)

#--------------------RQ:How much eth is lost from root_id to consolidation address--------------------
print(display_table_title("outflow_from_root_df"))
outflow_from_root_df = db.sql('''
SELECT root_id,
SUM(value_eth) AS sum_eth1
FROM foundation_table_df
GROUP BY root_id;''').df()
display(outflow_from_root_df)

print(display_table_title("inflow_to_consolidation_df"))
inflow_to_consolidation_df = db.sql('''
SELECT root_id,
SUM(value_eth) AS sum_eth2
FROM consolidated_address_depth_df
GROUP BY root_id;''').df()
display(inflow_to_consolidation_df)

print(display_table_title("eth_lost_df"))
eth_lost_df = db.sql('''
SELECT t1.root_id,
t1.sum_eth1 AS sum_eth_from_root,
t2.sum_eth2 AS sum_eth_to_consolidation,
ROUND((t1.sum_eth1 - t2.sum_eth2), 4) AS eth_lost,
ROUND(((t1.sum_eth1 - t2.sum_eth2)*100 / t1.sum_eth1), 2) AS percentage_of_eth_lost
FROM outflow_from_root_df t1
LEFT JOIN inflow_to_consolidation_df t2
ON t1.root_id = t2.root_id;''').df()
display(eth_lost_df)



## 6. Sanctioned Actors:
This section analyzes the flow of funds to various known entity types (e.g., sanctioned, mixers, CEX) across different hop depths.

In [ ]:
#Sanctioned addresses anywhere in hop chain---------------------

print(display_table_title("sanctioned_in_hops_df"))

sanctioned_in_hops_df = db.sql('''
SELECT
root_label,
root_id,
hop_depth,
from_address,
to_address,
to_entity_name,
to_entity_type,
from_entity_name,
from_entity_type,
Date,
tx_count,
ROUND(value_eth, 4) AS value_eth,
ROUND(USD_price_per_ETH, 2) AS USD_price_per_ETH,
FROM foundation_table_df
WHERE to_entity_type = 'SANCTIONED'
OR from_entity_type = 'SANCTIONED'
ORDER BY hop_depth, USD_price_per_ETH DESC
''').df()
display(sanctioned_in_hops_df)

sanctioned_in_hops_df_1 = db.sql('''
SELECT *
FROM sanctioned_in_hops_df
WHERE to_address IN ('0x4f47bc496083c727c5fbe3ce9cdf2b0f6496270c', '0x97b1043abd9e6fc31681635166d430a458d14f9c','0xb6f5ec1a0a9cd1526536d3f0426c429529471f40')
OR from_address IN ('0x4f47bc496083c727c5fbe3ce9cdf2b0f6496270c', '0x97b1043abd9e6fc31681635166d430a458d14f9c','0xb6f5ec1a0a9cd1526536d3f0426c429529471f40');''').df()
display(sanctioned_in_hops_df_1)


print(f"Sanctioned endpoints found:    {len(sanctioned_in_hops_df)}")
print(f"Unique sanctioned addresses:   {sanctioned_in_hops_df['to_address'].nunique()}")
print(f"Root wallets involved:         {sanctioned_in_hops_df['root_id'].nunique()}")


initial_payment_addresses_df = db.sql('''
SELECT *
FROM read_json('/content/sample_data/wallet_addr.json');''').df()
display(initial_payment_addresses_df)


#join the two tables---------------------

print(display_table_title("combined_sanctioned_addresses_df"))

combined_sanctioned_addresses_df = db.sql('''
SELECT t1.json,
t2.root_label,
t2.root_id,
t2.hop_depth,
t2.from_address,
t2.to_address,
t2.to_entity_name,
t2.to_entity_type,
t2.from_entity_name,
t2.from_entity_type,
t2.Date,
t2.tx_count,
t2.value_eth,
t2.USD_price_per_ETH,
FROM sanctioned_in_hops_df t2
LEFT JOIN initial_payment_addresses_df t1
ON lower(t1."json") = t2.from_address
OR lower(t1."json") = t2.to_address;''').df()
display(combined_sanctioned_addresses_df)

combined_sanctioned_addresses_df_1 = db.sql('''
SELECT to_address, hop_depth,
COUNT (DISTINCT to_entity_type) AS unique_entity_type,
FROM combined_sanctioned_addresses_df
WHERE to_entity_type = 'SANCTIONED'
GROUP BY to_address, hop_depth;''').df()
display(combined_sanctioned_addresses_df_1)

#Direction of sanctioned contact: This tells you whether sanctioned actors are being paid (receivers)
#or laundering onward (senders) — very different implications.

print(display_table_title("direction_sanctioned_df"))

direction_sanctioned_df = db.sql('''
SELECT
  CASE
    WHEN from_entity_type = 'SANCTIONED' AND to_entity_type = 'SANCTIONED' THEN 'both_sanctioned'
    WHEN from_entity_type = 'SANCTIONED' THEN 'sanctioned_sender'
    WHEN to_entity_type = 'SANCTIONED' THEN 'sanctioned_reciever'
    ELSE 'not_sanctioned_involved'
  END AS sanctioned_role,
  hop_depth,
  COUNT(*) AS count,
  SUM(USD_price_per_ETH) AS total_usd
  FROM combined_sanctioned_addresses_df
  GROUP BY sanctioned_role, hop_depth
  ORDER BY hop_depth;
  ''').df()
display(direction_sanctioned_df)


#Are the same sanctioned addresses reachable from multiple root_ids?

print(display_table_title("same_sanctioned_addresses_df"))

same_sanctioned_address = db.sql('''
SELECT
to_entity_name,
to_entity_type,
to_address,
COUNT(DISTINCT root_id) AS reached_from_n_roots,
MIN(hop_depth) AS earliest_hop,
SUM(USD_price_per_ETH) AS total_usd
FROM combined_sanctioned_addresses_df
WHERE to_entity_type = 'SANCTIONED'
GROUP BY to_entity_name, to_address, to_entity_type
ORDER BY reached_from_n_roots DESC;''').df()
display(same_sanctioned_address)


#Hop depth distribution

print(display_table_title("hop_depth_distribution_df"))

hop_depth_distribution_df = db.sql('''
SELECT
hop_depth,
COUNT(DISTINCT(root_id)) AS unique_root_ids,
COUNT(DISTINCT(to_address)) AS unique_sanctioned_addresses,
COUNT(DISTINCT(from_address)) AS unique_sender_addresses,
SUM(USD_price_per_ETH) AS total_usd,
FROM combined_sanctioned_addresses_df
GROUP BY hop_depth
ORDER BY hop_depth;''').df()
display(hop_depth_distribution_df)

sanity_check_df_1 = db.sql('''
SELECT root_label, COUNT(DISTINCT root_id) AS n_ids
FROM combined_sanctioned_addresses_df
GROUP BY root_label
HAVING n_ids > 1
''').df()
display(sanity_check_df_1)

In [ ]:
#---------------------VISUALISATION 1: bar chart of sanctioned addresse value by hop depth---------------------
from plotly.graph_objs import XAxis


vizC_df = db.sql('''
SELECT
hop_depth,
SUM(USD_price_per_ETH) AS USD_price_per_ETH,
FROM sanctioned_in_hops_df
GROUP BY hop_depth
ORDER BY hop_depth;''').df()
display(vizC_df)

vizC_sanctioned_df= db.sql('''
SELECT
hop_depth,
SUM(USD_price_per_ETH) AS USD_price_per_ETH,
SUM(tx_count) AS total_transactions,
COUNT(DISTINCT(to_address)) AS unique_sanctioned_addresses,

COUNT(DISTINCT(root_id)) AS unique_root_ids
FROM sanctioned_in_hops_df
WHERE to_entity_type = 'SANCTIONED'

GROUP BY hop_depth
ORDER BY hop_depth;''').df()
display(vizC_sanctioned_df)




import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


#---------------------CHART 1: SANCTION EXPOSURE - USD VALUE + ROOT WALLETS + UNIQUE ADDRESSES---------------------

fig = make_subplots(specs=[[{'secondary_y': True}]])

#USD value bars---------------------
fig.add_trace(
    go.Bar(
        x=vizC_sanctioned_df["hop_depth"],
        y=vizC_sanctioned_df["USD_price_per_ETH"],
        name="Total USD Value to Sanctioned",
        marker_color=[
            "#E74C3C" if v == vizC_sanctioned_df["USD_price_per_ETH"].max() #mark as red if it its max
            else "#2471A3"
            for v in vizC_sanctioned_df["USD_price_per_ETH"]
        ],
        text=[
            "$" + f"{v/1e6:.1f}M" if v>= 1e6 else "$" + f"{v/1e3:.0f}K"
            for v in vizC_sanctioned_df["USD_price_per_ETH"]
        ],
        textposition="outside",
        textfont=dict(size=10, color="#2C3E50"),
        customdata = vizC_sanctioned_df[["unique_sanctioned_addresses", "total_transactions"
        ]].values,
        hovertemplate=(
            "<b>Hop Depth %{x}</b><br>"
            "Total USD: $%{y:,.0f}<br>"
            "Unique Sanctioned Addresses: %{customdata[0]}<br>"
            "Total Transactions: %{customdata[1]}<br>"
            "<extra></extra>"
        )

      ),
    secondary_y=False
)

#Root wallets involved (overlayed as a scatter plot for clarity, or can be a separate bar)---------------------

fig.add_trace(
    go.Scatter(
        x=vizC_sanctioned_df["hop_depth"],
        y=vizC_sanctioned_df["unique_root_ids"],
        name="Root wallets involved",
        mode="lines+markers+text",
        yaxis="y2",
        line=dict(color="#2ECC71", width=2, dash="dash"),
        marker=dict(size=8, color="#2ECC71"),
        text= vizC_sanctioned_df["unique_root_ids"],
        textposition="top center",
        textfont=dict(size=10, color="#2ECC71"),
        hovertemplate = (
            "<b>Hop Depth %{x}</b><br>"
            "Root Wallets Involved: %{y}<br>"
            "<extra></extra>"
        )

    ),
    secondary_y=True
)


fig.update_layout(
    title = dict(
        text    = "Sanctioned Infrastructure Exposure Across Hop Depths",
        subtitle= dict(
            text = "USD value, root wallet involvement and unique sanctioned endpoints — DPRK ITW network"
        ),
        font    = dict(size=16, color="#2C3E50", family="Arial"),
        x       = 0.5,
        xanchor = "center"
    ),

    xaxis = dict(
        title_text = "Hop Depth",
        tickmode   = "linear",
        tick0      = 1,
        dtick      = 1,
        showgrid=False,
        linecolor= "#BDC3C7",
        type="category"
    ),


    yaxis = dict(
        title="Total USD Value",
        tickprefix = "$",
        showgrid=True,
        gridcolor="#ECF0F1",
        linecolor="#BDC3C7"
    ),

    yaxis2=dict(
        title="Root Wallets Involved",
        tickfont=dict(color="#2ECC71"),
        showgrid=False,
        linecolor="#BDC3C7"
    ),
    legend=dict(
        orientation="h",
        x=0.5,
        xanchor="center",
        y=-0.15,
        bgcolor="#FDFEFE",
        bordercolor="#BDC3C7",
        borderwidth=1
    ),
    paper_bgcolor = "#FDFEFE",
    plot_bgcolor  = "#FDFEFE",
    height        = 550,
    width         = 850,
    font          = dict(size=11, color="#2C3E50", family="Arial"),
    margin        = dict(t=110, b=100, l=80, r=80),
    showlegend    = True,
    annotations = [dict(
        text      = "The hop depth at which sanctioned addresses first appear indicates the minimum layering distance DPRK ITW workers maintain before reaching sanctioned infrastructure",
        xref      = "paper", yref="paper",
        x=0.5, y=-0.30,
        showarrow = False,
        font      = dict(size=10, color="#7F8C8D", family="Arial"),
        align     = "center"
    )]
)


fig.show()
fig.write_html('/content/sample_data/html/figure_9.html')



In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

#COLOUR MAP-------------------
role_colours = {
    "sanctioned_reciever": "#E74C3C",
    "sanctioned_sender":"#E67E22",
    "both_sanctioned":"#8E44AD"
}

#BUILDING SUBPLOTS

fig= make_subplots(
    rows=1,
    cols=1,
    subplot_titles=(
        "Sanctioned Entities: Root Wallet Reach"
    )
)


#CHART 2: HORIZONTAL BAR CHART (formerly the right subplot)

fig.add_trace(
    go.Bar(
        x=same_sanctioned_address["reached_from_n_roots"],
        y=same_sanctioned_address["to_entity_name"],
        orientation="h",
        marker_color=["#C0392B", "#E74C3C"],
        text=[
            f"${v:,.0f} | Earliest hop:{h}"
            for v, h in zip(
                same_sanctioned_address["total_usd"],
                same_sanctioned_address["earliest_hop"]
            )
        ],
        textposition="outside",
        hovertemplate=(
            "entity:%{y}<br>"
            "Reached from %{x} root wallets<br>"
            "<extra></extra>"
        )
    ),
    row=1, col=1
)

address_lookup = same_sanctioned_address.set_index("to_entity_name")["to_address"]


#LAYOUT

fig.update_layout (
    title = dict(
        text    = "Sanctioned Actor Exposure Across the DPRK ITW Network",
        subtitle= dict(
            text = "Reach of Sanctioned Entities by Number of Root Wallets"
        ),
        font = dict(size=16, color="#2C3E50", family= "Arial"),
        x=0.5,
        xanchor="center"
    ),
    paper_bgcolor = "#FDFEFE",
    plot_bgcolor  = "#FDFEFE",
    height=580,
    width=800,
    font=dict(size=11, color="#2C3E50", family="Arial"),
    margin=dict(t=110, b=190, l=60, r=80),
    legend=dict(
        orientation="h",
        x=0.0,
        y=-0.18,
        font=dict(size=10)
    )
)


#AXIS FORMATTING

fig.update_xaxes(
    title_text="Number of Root Wallets",
    range=[0,38],
    showgrid=False,
    linecolor="#BDC3C7",
    row=1, col=1
)

fig.update_yaxes(
    showgrid=False,
    linecolor="#BDC3C7",
    row=1, col=1
)


for i, entity in enumerate(same_sanctioned_address["to_entity_name"]):
  addr=address_lookup[entity]
  short_addr = f"{addr[:6]}...{addr[-4:]}"

  fig.add_annotation(
      xref="x",
      yref="y",
      x=0.5,
      y=entity,
      text=f"<i>{short_addr}</i>",
      showarrow=False,
      font=dict(size=15, color="#FFFFFF", family = "Arial"),
      xanchor="left",
      yanchor="top",
      yshift=-12
  )

fig.show()
fig.write_html('/content/sample_data/html/figure_8.html')


##7. Behaviour of other labeled entities:
Grouping by identity type

In [ ]:
vizC_entity_type_df= db.sql('''
SELECT
hop_depth,
to_entity_type,
SUM(USD_price_per_ETH) AS USD_price_per_ETH,
SUM(tx_count) AS total_transactions,
COUNT(DISTINCT(to_address)) AS unique_addresses,
COUNT(DISTINCT(root_id)) AS unique_root_ids
FROM foundation_table_df
WHERE to_entity_type != 'UNATTRIBUTED'
AND USD_price_per_ETH IS NOT NULL
GROUP BY hop_depth, to_entity_type
ORDER BY hop_depth, to_entity_type;''').df()
display(vizC_entity_type_df)

print(vizC_entity_type_df[vizC_entity_type_df["to_entity_type"] == "DEX"])


#--------------------CHART 2: ENTITY TYPE SUBPLOTS: ORDERED BY RISK---------------------

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd


#order by risk level, with highest risk first---------------------
entity_order  = ["SANCTIONED", "MIXER", "BRIDGE", "CEX", "DEX"]

#Colour per entity type — red for high risk, graduating to lower risk---------------------
colour_map = {
    "SANCTIONED": "#E74C3C",
    "MIXER":      "#E67E22",
    "BRIDGE":     "#F1C40F",
    "CEX":        "#2471A3",
    "DEX":        "#1A5276"
}


#Filter to only entity_types that actually exist in the dataset---------------------

# entity_types=[
#     e for e in entity_order
#     if e in vizC_entity_type_df["to_entity_type"].unique()
# ]
entity_types = (vizC_entity_type_df["to_entity_type"].unique()).tolist()
n = len(entity_types)

#1 row if 3 or fewer and 2 rows if more---------------------

rows=1 if n<=3 else 2
cols=3


fig2 = make_subplots(
    rows               = rows,
    cols               = cols,
    subplot_titles     = entity_types,
    horizontal_spacing = 0.12,
    vertical_spacing   = 0.15
)

#One subplot per entity type ---------------------
for i, entity in enumerate(entity_types):
    row = i // cols + 1
    col = i %  cols + 1

    subset = vizC_entity_type_df[vizC_entity_type_df["to_entity_type"] == entity].copy()

    if subset.empty:

        fig2.add_trace(
            go.Bar(x=[], y=[], showlegend=False),
            row=row, col=col
        )
        continue

    #USD bars---------------------
    fig2.add_trace(
        go.Bar(
            x            = subset["hop_depth"],
            y            = subset["USD_price_per_ETH"],
            name         = entity,
            marker_color = colour_map[entity],
            opacity      = 0.85,
            showlegend   = False,
            text         = ["$" + f"{v/1e6:.2f}M" if v >= 1e6
                            else "$" + f"{v/1e3:.0f}K"
                            for v in subset["USD_price_per_ETH"]],
            textposition = "outside",
            textfont     = dict(size=9, color="#2C3E50"),
            customdata= subset[["unique_root_ids", "unique_addresses"]].values,
            hovertemplate=(
                f"<b>{entity}</b><br>"
                "Hop Depth: %{x}<br>"
                "USD Value: $%{y:,.0f}<br>"
                "Root Wallets: %{customdata[0]}<br>"
                "Unique Addresses: %{customdata[1]}<br>"
                "<extra></extra>"
            )
        ),
        row=row, col=col
    )

    #Root wallets as dot overlay---------------------
    y_base = subset["USD_price_per_ETH"].max() * 0.05
    y_base = y_base if y_base > 0 else 0

    fig2.add_trace(
        go.Scatter(
            x          = subset["hop_depth"],
            y          = [y_base] * len(subset),
            mode       = "markers+text",
            marker     = dict(
                size   = subset["unique_root_ids"] * 4,
                color  = colour_map[entity],
                opacity= 0.35,
                line   = dict(color="white", width=1)
            ),
            text         = subset["unique_root_ids"],
            textposition = "middle center",
            textfont   = dict(size=8, color="#7F8C8D"),
            showlegend = False,
            hovertemplate = (
                f"<b>{entity}</b><br>"
                "Hop: %{x}<br>"
                "Root Wallets: %{text}<br>"
                "<extra></extra>"
            )
        ),
        row=row, col=col
    )
    #Consistent axis formatting across all subplots---------------------

    fig2.update_xaxes(
        title_text = "Hop Depth",
        tickmode   = "linear",
        tick0       = 1,
        dtick      = 1,
        showgrid   = True,
        linecolor  = "#BDC3C7",
        type       = "category",
        row=row, col=col
    )
    fig2.update_yaxes(
        title_text = "Total USD",
        tickprefix="$",
        showgrid   = True,
        gridcolor  = "#ECF0F1",
        linecolor  = "#BDC3C7",
        automargin=True,
        row=row, col=col
    )
    fig2.update_yaxes(rangemode="nonnegative")


fig2.update_layout(
    title = dict(
        text    = "Fund Flow to Known Infrastructure by Entity Type",
        subtitle= dict(
            text = "Ordered by risk level · bars = USD value · dot size = root wallets involved · DPRK ITW network - UNATTRIBUTED addresses excluded "
        ),
        font    = dict(size=16, color="#2C3E50", family="Arial"),
        x        = 0.5,
        xanchor = "center"
    ),
    paper_bgcolor = "#FDFEFE",
    plot_bgcolor   = "#FDFEFE",
    height        = 500*rows,
    width         = 1050,
    font          = dict(size=10, color="#2C3E50", family="Arial"),
    margin        = dict(t=140, b=90, l=70, r=50)
)

fig2.add_annotation(
    text      = (
        "Sanctioned and Mixer exposure indicates deliberate obfuscation. "
        "CEX and DEX activity suggests eventual conversion to fiat or other assets."
    ),
    xref="paper", yref="paper",
    x=0.5, y=-0.15,
    showarrow=False,
    font=dict(size=8, color="#7F8C8D", family="Arial"),
    align="center"
)


for ann in fig2.layout.annotations:
    clean_text = ann.text.replace("<b>", "").replace("</b>", "").strip()
    if clean_text in entity_types:
        ann.update(
            font=dict(size=10, color=colour_map.get(clean_text, "#2C3E50"), family="Arial"),
            text=f"<b>{clean_text}</b>"
        )



fig2.show()
fig2.write_html('/content/sample_data/html/figure_6.html')


## 8. Traceability: How much value disappears into unattributed addresses

In [ ]:
#--------------------TRACEBILITY PERCENTAGE: WHAT IS INVISIBLE AND WHAT ISN'T---------------------

tracability_percentage_df= db.sql('''
WITH hop_totals AS (
    SELECT
        hop_depth,
        SUM(USD_price_per_ETH) AS total_usd
    FROM foundation_table_df
    GROUP BY hop_depth
),
hop_attributed AS (
    SELECT
        hop_depth,
        SUM(USD_price_per_ETH) AS attributed_usd
    FROM foundation_table_df
    WHERE to_entity_type != 'UNATTRIBUTED'
    GROUP BY hop_depth
)
SELECT
    t.hop_depth,
    t.total_usd,
    a.attributed_usd,
    ROUND((a.attributed_usd / t.total_usd) * 100, 2) AS pct_traceable,
    ROUND(((t.total_usd - a.attributed_usd) / t.total_usd) * 100, 2) AS pct_lost_to_unattributed
FROM hop_totals t
LEFT JOIN hop_attributed a ON t.hop_depth = a.hop_depth
ORDER BY t.hop_depth;''').df()
display(tracability_percentage_df)

import plotly.graph_objects as go
from plotly.subplots import make_subplots

#Data
hops = [1, 2, 3, 4]
total_usd = [8.80, 62.52, 150.30, 433.86]
pct_traceable = [1.31, 41.10, 11.76, 2.68]
pct_lost = [98.69, 58.90, 88.24, 97.32]

#----- CHART 1: Traceability % stacked bar -----
fig1 = make_subplots(specs=[[{"secondary_y": False}]])

fig1.add_trace(go.Bar(
    x=hops,
    y=pct_lost,
    name="Lost to unattributed addresses",
    marker_color="#BDC3C7",
    text=[f"{v:.2f}%" for v in pct_lost],
    textposition="inside",
    textfont=dict(size=10, color="#7F8C8D", family="Arial"),
    hovertemplate=(
        "<b>Hop Depth %{x}</b><br>"
        "Unattributed: %{y:.2f}%<br>"
        "<extra></extra>"
    )
))

fig1.add_trace(go.Bar(
    x=hops,
    y=pct_traceable,
    name="Traceable (attributed)",
    marker_color=[
        "#E74C3C" if v == max(pct_traceable)
        else "#2471A3"
        for v in pct_traceable
    ],
    text=[f"{v:.2f}%" for v in pct_traceable],
    textposition="outside",
    textfont=dict(size=10, color="#2C3E50", family="Arial"),
    hovertemplate=(
        "<b>Hop Depth %{x}</b><br>"
        "Traceable: %{y:.2f}%<br>"
        "<extra></extra>"
    )
))

fig1.update_layout(
    barmode="stack",
    title=dict(
        text="Traceability of DPRK ITW Funds Across Hop Depths",
        subtitle=dict(
            text="Percentage of total USD value attributed to labelled vs unattributed addresses — DPRK ITW network"
        ),
        font=dict(size=16, color="#2C3E50", family="Arial"),
        x=0.5,
        xanchor="center"
    ),
    xaxis=dict(
        title_text="Hop Depth",
        tickmode="linear",
        tick0=1,
        dtick=1,
        showgrid=False,
        linecolor="#BDC3C7",
        type="category"
    ),
    yaxis=dict(
        title="Percentage (%)",
        range=[0, 115],
        showgrid=True,
        gridcolor="#ECF0F1",
        linecolor="#BDC3C7",
        ticksuffix="%"
    ),
    legend=dict(
        orientation="h",
        x=0.5,
        xanchor="center",
        y=-0.15,
        bgcolor="#FDFEFE",
        bordercolor="#BDC3C7",
        borderwidth=1
    ),
    paper_bgcolor="#FDFEFE",
    plot_bgcolor="#FDFEFE",
    height=550,
    width=850,
    font=dict(size=11, color="#2C3E50", family="Arial"),
    margin=dict(t=110, b=120, l=80, r=80),
    showlegend=True,
    annotations=[dict(
        text="Hop 2 traceability spike driven by sanctioned entity concentration ($25.6M across 3 addresses). ETH transfers only — USDT/USDC flows excluded.",
        xref="paper", yref="paper",
        x=0.5, y=-0.35,
        showarrow=False,
        font=dict(size=10, color="#7F8C8D", family="Arial"),
        align="center"
    )]
)

metrics = [
    ("Hop 1", "$8.8M", "1.31% traceable"),
    ("Hop 2", "$62.5M", "41.10% traceable"),
    ("Hop 3", "$150.3M", "11.76% traceable"),
    ("Hop 4", "$433.9M", "2.68% traceable"),
]

x_positions = [0.05, 0.30, 0.62, 0.95]

for (label, value, sub), x in zip(metrics, x_positions):
    fig1.add_annotation(
        text=f"<b>{label}</b><br>{value}<br><span style='color:#7F8C8D'>{sub}</span>",
        xref="paper", yref="paper",
        x=x, y=0.2,
        showarrow=False,
        font=dict(size=11, color="#2C3E50", family="Arial"),
        align="center",
        bgcolor="#ECF0F1",
        bordercolor="#BDC3C7",
        borderwidth=1,
        borderpad=6
    )


fig1.show()
fig1.write_html('/content/sample_data/html/figure_7.html')

